# Chapter 13: Network Documentation
## From Manual Failure to Automated Knowledge — Volume 2 Begins

**Welcome to Volume 2.** Volume 1 built the foundations (API, prompts, costs, testing).
Volume 2 applies them to real network engineering problems. We start with the one that
causes the most pain: documentation.

**What you'll build — 5 demos:**
1. **Two-stage AI pipeline** — Haiku extracts facts, Sonnet writes the docs
2. **Config fingerprinting** — hash-based change detection (OSPF sequence number analogy)
3. **Multi-audience synthesis** — same config → ops doc, management summary, security report
4. **Auto change log** — compare two configs, get human-readable changelog
5. **Mini network wiki** — query your docs with plain English questions

**The core insight:**
> Documentation fails because we treat it as a deliverable — written once, immediately drifting.
> OSPF doesn't write its routing table once. It recomputes it automatically whenever topology changes.
> Documentation should work the same way: **computed from the actual state, not written manually.**

**Network analogy for this chapter:**
| OSPF Mechanism | Documentation Equivalent |
|---|---|
| Hello (detect change) | Config hash comparison |
| LSA exchange (collect state) | Config collection from devices |
| SPF calculation | AI documentation synthesis |
| Routing table (output) | Generated markdown documentation |
| Sequence number (change detect) | SHA-256 config fingerprint |

**No real devices needed** — all configs are realistic strings. Runs entirely in Colab.


In [ ]:
!pip install anthropic --quiet
print("Ready!")


In [ ]:
import os, json, re, hashlib
from anthropic import Anthropic

# API key: Colab secrets first, getpass fallback for local/other environments
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"   # extraction: fast + cheap
SONNET = "claude-sonnet-4-20250514"    # synthesis: better writing quality

def ask(prompt, model=HAIKU, max_tokens=800, system=None, temperature=0):
    kwargs = dict(model=model, max_tokens=max_tokens, temperature=temperature,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    resp = client.messages.create(**kwargs)
    return resp.content[0].text

print("Setup complete.")
print(f"HAIKU (extraction): {HAIKU}")
print(f"SONNET (synthesis) : {SONNET}")


## The 5 Documentation Failure Modes

Every network team suffers from these. Sound familiar?

| Failure Mode | What it looks like | Cost |
|---|---|---|
| **Drift** | Docs existed, never updated after changes | Wrong during every incident |
| **Absence** | "Quick" changes with no docs. Since 2019. | Pure tribal knowledge |
| **Tribal knowledge** | Only one engineer knows this BGP policy | Catastrophic when they leave |
| **Fragmentation** | Real info split across Confluence, email, chat, Jira | Hours to find the right piece |
| **Audience mismatch** | 200-line config pasted into Confluence | Not documentation, it's a dump |

**AI fixes all five — not by magic, but through automation.**
Automation is the only reliable solution for stage 5 of the documentation lifecycle (maintenance).

---

## Demo 1: Two-Stage AI Documentation Pipeline

**Why two stages?**

| Stage | Model | Task | Why |
|---|---|---|---|
| Extract | Haiku | Config → structured JSON | Fast, cheap, good enough for parsing |
| Synthesize | Sonnet | JSON + config → readable docs | Better writing quality for human readers |

This mirrors OSPF: use lightweight Hellos for frequent checks (Haiku),
heavy SPF calculation only when needed (Sonnet).


In [ ]:
# Realistic network configs — two devices in a hub-spoke design

CORE_ROUTER_CONFIG = """hostname core-rtr-01
!
version 15.4
!
interface GigabitEthernet0/0
 description [UPLINK] To-ISP-Primary
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description [DOWNLINK] To-Distribution-SW-01
 ip address 10.0.0.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/2
 description [DOWNLINK] To-Distribution-SW-02
 ip address 10.0.0.5 255.255.255.252
 no shutdown
!
interface Loopback0
 description Router-ID
 ip address 10.255.0.1 255.255.255.255
!
router ospf 1
 router-id 10.255.0.1
 network 10.0.0.0 0.255.255.255 area 0
 passive-interface GigabitEthernet0/0
!
router bgp 65001
 bgp router-id 10.255.0.1
 neighbor 203.0.113.2 remote-as 9876
 neighbor 203.0.113.2 description ISP-Transit-BGP
 !
 address-family ipv4
  network 198.51.100.0 mask 255.255.255.0
  neighbor 203.0.113.2 activate
  neighbor 203.0.113.2 soft-reconfiguration inbound
 exit-address-family
!
ip access-list extended BLOCK_MANAGEMENT
 deny   tcp any any eq 23
 deny   tcp any any eq 21
 permit ip any any
!
ntp server 216.239.35.0
ntp server 216.239.35.4
!
logging 10.0.100.5
logging buffered 64000"""

BRANCH_ROUTER_CONFIG = """hostname branch-rtr-07
!
interface GigabitEthernet0/0
 description [UPLINK] MPLS-to-Core via provider
 ip address 172.16.7.2 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description [LAN] Branch-Office-LAN
 ip address 192.168.7.1 255.255.255.0
 ip helper-address 10.0.100.10
 no shutdown
!
router ospf 1
 router-id 172.16.7.2
 network 172.16.7.0 0.0.0.3 area 1
 network 192.168.7.0 0.0.0.255 area 1
 area 1 stub
!
ip route 0.0.0.0 0.0.0.0 172.16.7.1
!
ntp server 10.255.0.1"""

print("Sample configs loaded:")
print(f"  core-rtr-01  : {len(CORE_ROUTER_CONFIG)} chars")
print(f"  branch-rtr-07: {len(BRANCH_ROUTER_CONFIG)} chars")


In [ ]:
def extract_device_facts(hostname, config):
    """
    Stage 1 — Haiku: extract structured JSON facts from raw config.
    Fast and cheap. Run on every device in your fleet.
    """
    prompt = f"""Extract key facts from this Cisco IOS config as JSON.
Device: {hostname}

Config:
{config[:3000]}

Return ONLY valid JSON with these exact keys:
{{
  "hostname": "...",
  "device_role": "core-router|distribution-switch|access-switch|branch-router|firewall",
  "routing_protocols": ["ospf", "bgp", ...],
  "bgp_asn": null,
  "ospf_areas": [],
  "interface_count": 0,
  "has_acl": true,
  "uplink_interfaces": ["..."],
  "lan_interfaces": ["..."],
  "one_line_summary": "one sentence describing this device"
}}"""

    text = ask(prompt, model=HAIKU, max_tokens=500, temperature=0)
    m    = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        return json.loads(m.group())
    return {"hostname": hostname, "device_role": "unknown", "one_line_summary": "parse failed"}

def synthesize_documentation(hostname, facts, config, audience="operations"):
    """
    Stage 2 — Sonnet: generate human-readable documentation.
    Better writing quality. Run only when config has changed (see Demo 2).
    """
    audience_instructions = {
        "operations": "Write for network operations engineers. Focus on: what this device does, how to troubleshoot it, what commands to run for common problems.",
        "management": "Write for IT management. No CLI commands. Focus on: business function, what breaks if this device fails, risk level.",
        "security":   "Write for the security team. Focus on: what traffic is permitted/denied, exposed services, authentication methods, ACL coverage.",
    }
    instruction = audience_instructions.get(audience, audience_instructions["operations"])

    prompt = f"""{instruction}

Device: {hostname}
Extracted facts: {json.dumps(facts, indent=2)}

Relevant config sections:
{config[:2500]}

Write structured documentation in Markdown format.
Include a one-line summary at the top, then key sections relevant to the audience.
Keep it concise — engineers are busy. Max 400 words."""

    return ask(prompt, model=SONNET, max_tokens=600, system="You are a senior network engineer writing documentation.", temperature=0)

# Run the two-stage pipeline
print("Stage 1 — Extracting facts (Haiku)...")
facts = extract_device_facts("core-rtr-01", CORE_ROUTER_CONFIG)

print("\nExtracted facts:")
for k, v in facts.items():
    print(f"  {k:20s}: {v}")

print("\nStage 2 — Synthesizing documentation (Sonnet)...")
doc = synthesize_documentation("core-rtr-01", facts, CORE_ROUTER_CONFIG, audience="operations")
print("\n" + "="*60)
print(doc)


## Demo 2: Config Fingerprinting — Change Detection

**The incremental update problem:**
Running a full documentation pass over 500 devices is expensive.
But if you run it infrequently, docs drift.

**The solution: config fingerprinting** — the same idea as OSPF LSA sequence numbers.

```
OSPF:                          Documentation:
  Seq number per LSA    →        SHA-256 hash per device config
  Compare seqnums       →        Compare current hash vs stored hash
  Recalculate SPF only  →        Re-generate docs only when hash differs
    when seq changed
```

A typical enterprise network has 5-10% of devices change in any period.
Fingerprinting reduces documentation API cost by **~90%** — you only pay to
re-document the devices that actually changed.

**The `DocCache` class** maintains:
- `{hostname → last_hash}` — the stored fingerprint
- `{hostname → last_doc}` — the cached documentation


In [ ]:
class DocCache:
    """
    Hash-based documentation cache.
    Only re-generates docs when config actually changes.
    Network analogy: OSPF only runs SPF when LSA sequence number changes.
    """

    def __init__(self):
        self._hashes = {}   # hostname → SHA-256 of last config
        self._docs   = {}   # hostname → last generated documentation
        self._stats  = {"generated": 0, "cached": 0, "cost_saved": 0}

    def config_hash(self, config):
        """SHA-256 fingerprint of the config. Same config = same hash."""
        return hashlib.sha256(config.encode()).hexdigest()[:16]

    def needs_update(self, hostname, config):
        """True if config changed since last documentation run."""
        current_hash = self.config_hash(config)
        stored_hash  = self._hashes.get(hostname)
        return current_hash != stored_hash

    def get_or_generate(self, hostname, config, audience="operations"):
        """
        Return cached doc if config is unchanged.
        Generate new doc (and cache it) if config changed.
        """
        if not self.needs_update(hostname, config):
            self._stats["cached"] += 1
            self._stats["cost_saved"] += 1   # approximate: 1 API call saved
            return self._docs[hostname], "cached"

        # Config changed (or first time) — generate new documentation
        facts = extract_device_facts(hostname, config)
        doc   = synthesize_documentation(hostname, facts, config, audience=audience)

        # Store hash and doc
        self._hashes[hostname] = self.config_hash(config)
        self._docs[hostname]   = doc
        self._stats["generated"] += 1
        return doc, "generated"

    def stats(self):
        total = self._stats["generated"] + self._stats["cached"]
        if total == 0:
            return self._stats
        saved_pct = self._stats["cached"] / total * 100
        return {**self._stats, "total_runs": total, "cache_hit_rate": f"{saved_pct:.0f}%"}


# Demo — show fingerprinting in action
cache = DocCache()

print("=== Run 1: First documentation pass (no cache) ===")
doc1, source1 = cache.get_or_generate("core-rtr-01", CORE_ROUTER_CONFIG)
print(f"  core-rtr-01:  {source1} (hash: {cache.config_hash(CORE_ROUTER_CONFIG)})")
doc2, source2 = cache.get_or_generate("branch-rtr-07", BRANCH_ROUTER_CONFIG)
print(f"  branch-rtr-07: {source2} (hash: {cache.config_hash(BRANCH_ROUTER_CONFIG)})")

print("\n=== Run 2: Same configs, nothing changed ===")
_, s1 = cache.get_or_generate("core-rtr-01",   CORE_ROUTER_CONFIG)
_, s2 = cache.get_or_generate("branch-rtr-07", BRANCH_ROUTER_CONFIG)
print(f"  core-rtr-01:   {s1}  ✅ no API call needed")
print(f"  branch-rtr-07: {s2}  ✅ no API call needed")

print("\n=== Run 3: Simulate a change on branch-rtr-07 ===")
BRANCH_MODIFIED = BRANCH_ROUTER_CONFIG.replace("area 1 stub", "area 1 totally-stub")
_, s1 = cache.get_or_generate("core-rtr-01",   CORE_ROUTER_CONFIG)
_, s3 = cache.get_or_generate("branch-rtr-07", BRANCH_MODIFIED)
print(f"  core-rtr-01:   {s1}  ✅ unchanged — cached")
print(f"  branch-rtr-07: {s3}  🔄 config changed — re-generated")

print(f"\nCache statistics: {cache.stats()}")


## Demo 3: Multi-Audience Documentation

**The audience mismatch problem:**
> A 200-line config pasted into Confluence is not documentation. It's a dump.

The same device serves different audiences who need completely different information:

| Audience | What they care about | What they don't need |
|---|---|---|
| **Operations** | CLI commands, troubleshooting steps, interface states | Business context |
| **Management** | Business impact, risk, what breaks in an outage | CLI syntax |
| **Security** | ACLs, exposed services, authentication, VPN config | OSPF timers |

With AI, you generate **one extraction** (Haiku, cheap) and **three syntheses** (Sonnet, one per audience).
No human writes three versions of the same documentation. AI does.

**Cost calculation:**
- Traditional: 3 engineers × 2 hours × $100/hr = **$600 per device**
- AI pipeline: ~$0.05 in API calls + 30 seconds = **$0.05 per device**


In [ ]:
def generate_all_audiences(hostname, config):
    """
    One extraction → three audience-specific documentation pages.
    Haiku runs once, Sonnet runs three times with different instructions.
    """
    print(f"Extracting facts for {hostname} (Haiku)...")
    facts = extract_device_facts(hostname, config)

    docs = {}
    audiences = ["operations", "management", "security"]

    for audience in audiences:
        print(f"Synthesizing {audience} docs (Sonnet)...")
        docs[audience] = synthesize_documentation(hostname, facts, config, audience=audience)

    return facts, docs

def print_doc_section(title, doc_text, max_chars=600):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(doc_text[:max_chars])
    if len(doc_text) > max_chars:
        print(f"... [{len(doc_text) - max_chars} more chars]")

# Generate all three audience docs for core-rtr-01
facts, docs = generate_all_audiences("core-rtr-01", CORE_ROUTER_CONFIG)

print_doc_section("OPERATIONS DOCUMENTATION", docs["operations"])
print_doc_section("MANAGEMENT SUMMARY", docs["management"])
print_doc_section("SECURITY REPORT", docs["security"])

# Show the difference in focus
print("\n" + "="*60)
print("KEY INSIGHT: same device config, three completely different documents")
print(f"  Operations doc: {len(docs['operations'])} chars")
print(f"  Management doc: {len(docs['management'])} chars")
print(f"  Security doc  : {len(docs['security'])} chars")


## Demo 4: Automated Change Log

**The absence failure mode, fixed:**
> Every change should generate documentation automatically. But nobody writes changelogs at midnight.

**How it works:**
1. Before the change: collect config hash (baseline)
2. Apply the change (human or automation)
3. After the change: collect new config
4. AI compares the two configs → generates human-readable changelog

This is what change management tools like RANCID/Oxidized do for change tracking,
plus AI to explain what the change *means* — not just what lines changed.

**The output should answer:**
- What configuration changed?
- What is the operational impact?
- Is this a risky change or routine?
- Who should review this?


In [ ]:
def generate_change_log(hostname, config_before, config_after):
    """
    AI-powered change log generator.
    Compares two configs and produces a human-readable changelog with impact assessment.
    """
    if config_before == config_after:
        return {"changed": False, "summary": "No changes detected"}

    # Create a simple diff (lines added/removed)
    lines_before = set(config_before.splitlines())
    lines_after  = set(config_after.splitlines())
    added   = [l for l in lines_after   if l not in lines_before and l.strip()]
    removed = [l for l in lines_before  if l not in lines_after  and l.strip()]

    diff_summary = f"Lines added ({len(added)}):\n"
    for l in added[:15]:
        diff_summary += f"  + {l}\n"
    diff_summary += f"\nLines removed ({len(removed)}):\n"
    for l in removed[:15]:
        diff_summary += f"  - {l}\n"

    prompt = f"""You are a senior network engineer reviewing a config change.
Device: {hostname}

CONFIG DIFF:
{diff_summary}

Write a change log entry with these sections:
1. Summary (1 sentence: what changed)
2. Operational Impact (what does this change to network behavior?)
3. Risk Level: LOW / MEDIUM / HIGH / CRITICAL
4. Rollback: what command(s) would undo this change
5. Review Required: YES (if High/Critical) or NO

Be specific and technical. Keep it under 200 words."""

    analysis = ask(prompt, model=SONNET, max_tokens=400, temperature=0)

    return {
        "changed":    True,
        "hostname":   hostname,
        "lines_added":   len(added),
        "lines_removed": len(removed),
        "diff_preview":  diff_summary[:400],
        "change_log":    analysis,
    }

# Demo — simulate three common change scenarios
print("=== Change 1: Adding BFD for BGP fast failover ===")
config_v2 = CORE_ROUTER_CONFIG.replace(
    "neighbor 203.0.113.2 description ISP-Transit-BGP",
    "neighbor 203.0.113.2 description ISP-Transit-BGP\n neighbor 203.0.113.2 fall-over bfd\n neighbor 203.0.113.2 timers 5 15"
)
cl1 = generate_change_log("core-rtr-01", CORE_ROUTER_CONFIG, config_v2)
print(cl1["change_log"])

print("\n\n=== Change 2: Removing security ACL (risky!) ===")
config_v3 = CORE_ROUTER_CONFIG.replace(
    "ip access-list extended BLOCK_MANAGEMENT\n deny   tcp any any eq 23\n deny   tcp any any eq 21\n permit ip any any\n!", ""
)
cl2 = generate_change_log("core-rtr-01", CORE_ROUTER_CONFIG, config_v3)
print(cl2["change_log"])


## Demo 5: Mini Network Wiki — Query Your Docs

**The fragmentation failure mode, fixed.**

Instead of searching through 500 Confluence pages, you ask a question in plain English.
The wiki finds the relevant documentation and answers from it.

**This is the foundation of Chapter 14 (RAG)** — we build the simple version here,
and the full production version in the next chapter.

**How it works:**
1. `NetworkWiki` stores device docs as indexed text
2. On query: keyword search finds relevant devices
3. Claude reads those docs and answers the question
4. Answer includes source citations

```
Query: "which devices run BGP and who are their peers?"
   ↓
Find docs for BGP devices (keyword match)
   ↓
Claude reads those docs + answers
   ↓
"core-rtr-01 runs eBGP with ISP AS 9876 via 203.0.113.2 [Source: core-rtr-01 ops doc]"
```


In [ ]:
class NetworkWiki:
    """
    Simple network documentation wiki with plain-English query.
    Stores device docs and answers questions from them.
    Chapter 14 replaces the keyword search with real vector embeddings.
    """

    def __init__(self):
        self.docs    = {}   # hostname → {ops: ..., management: ..., security: ...}
        self.facts   = {}   # hostname → extracted facts dict

    def add_device(self, hostname, config, audiences=None):
        """Extract facts and generate docs for this device."""
        audiences = audiences or ["operations"]
        print(f"  Indexing {hostname}...")
        self.facts[hostname] = extract_device_facts(hostname, config)
        self.docs[hostname]  = {}
        for audience in audiences:
            self.docs[hostname][audience] = synthesize_documentation(
                hostname, self.facts[hostname], config, audience=audience
            )

    def _find_relevant(self, query, top_k=3):
        """
        Simple keyword search across all stored docs.
        Chapter 14 replaces this with vector similarity search.
        """
        query_words = set(query.lower().split())
        scores = {}
        for hostname, doc_dict in self.docs.items():
            all_text = " ".join(doc_dict.values()).lower()
            # Score = number of query words found in the doc
            score = sum(1 for w in query_words if w in all_text)
            if score > 0:
                scores[hostname] = score
        # Return top_k by score
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [h for h, _ in ranked[:top_k]]

    def ask(self, question):
        """
        Answer a plain-English question from the documentation.
        """
        relevant_hosts = self._find_relevant(question)
        if not relevant_hosts:
            return {"answer": "No relevant documentation found.", "sources": []}

        # Build context from relevant docs
        context_parts = []
        for h in relevant_hosts:
            ops_doc = self.docs[h].get("operations", "")
            context_parts.append(f"[Device: {h}]\n{ops_doc[:800]}")
        context = "\n\n---\n\n".join(context_parts)

        prompt = f"""You are a network documentation assistant.
Answer this question using ONLY the documentation provided below.
Cite your sources using [Device: hostname] notation.

QUESTION: {question}

DOCUMENTATION:
{context}

Give a concise, accurate answer with source citations. If the answer is not in the docs, say so."""

        answer = ask(prompt, model=SONNET, max_tokens=400, temperature=0)
        return {
            "question": question,
            "answer":   answer,
            "sources":  relevant_hosts,
        }

    def inventory(self):
        """Print a summary of all indexed devices."""
        print(f"Network Wiki — {len(self.docs)} devices indexed")
        for hostname, facts_data in self.facts.items():
            role     = facts_data.get("device_role", "unknown")
            protocols = facts_data.get("routing_protocols", [])
            summary  = facts_data.get("one_line_summary", "")[:60]
            print(f"  {hostname:20s}  [{role}]  protocols: {protocols}")
            print(f"    {summary}")


# Build the wiki with both devices
print("Building network wiki...\n")
wiki = NetworkWiki()
wiki.add_device("core-rtr-01",   CORE_ROUTER_CONFIG,   audiences=["operations", "security"])
wiki.add_device("branch-rtr-07", BRANCH_ROUTER_CONFIG, audiences=["operations"])

print()
wiki.inventory()


In [ ]:
# Run plain-English queries against the wiki
queries = [
    "Which devices run BGP and who are the BGP peers?",
    "What OSPF areas are configured and what type are they?",
    "Which devices have security ACLs configured?",
    "If the core router fails, which branch sites are affected?",
]

print("Network Wiki Query Demo\n")
for q in queries:
    print(f"Q: {q}")
    result = wiki.ask(q)
    print(f"A: {result['answer'].strip()[:300]}")
    print(f"   Sources: {result['sources']}")
    print()


In [ ]:
# ── Putting it all together: the full documentation pipeline ──────────────

class NetworkDocPipeline:
    """
    Production-ready documentation pipeline:
    - Hash-based change detection (no redundant API calls)
    - Two-stage extraction + synthesis
    - Multi-audience output
    - Queryable wiki
    - Change log generation
    """

    def __init__(self):
        self.cache = DocCache()
        self.wiki  = NetworkWiki()
        self.changes = []   # log of all config changes detected

    def process_device(self, hostname, config, prev_config=None):
        """
        Process one device through the full pipeline.
        - If prev_config provided: generate change log
        - If config unchanged since last run: skip (use cache)
        - If new/changed: generate docs, update wiki
        """
        # Change log
        if prev_config and prev_config != config:
            cl = generate_change_log(hostname, prev_config, config)
            self.changes.append(cl)
            print(f"  [{hostname}] Config change detected — change log generated")

        # Doc generation (with caching)
        _, source = self.cache.get_or_generate(hostname, config)
        if source == "generated":
            # Update wiki with fresh docs
            self.wiki.add_device(hostname, config, audiences=["operations"])
            print(f"  [{hostname}] Docs generated and wiki updated")
        else:
            print(f"  [{hostname}] No change — using cached docs")

    def query(self, question):
        return self.wiki.ask(question)

    def report(self):
        print("\nPipeline Report:")
        print(f"  Cache stats : {self.cache.stats()}")
        print(f"  Changes logged: {len(self.changes)}")
        for c in self.changes:
            print(f"    → {c.get('hostname')}: +{c.get('lines_added',0)} -{c.get('lines_removed',0)} lines")
        print(f"  Wiki devices: {len(self.wiki.docs)}")


# Run the pipeline
print("Running full documentation pipeline...\n")
pipeline = NetworkDocPipeline()

pipeline.process_device("core-rtr-01",   CORE_ROUTER_CONFIG)
pipeline.process_device("branch-rtr-07", BRANCH_ROUTER_CONFIG)

# Simulate a change on branch-rtr-07
BRANCH_UPDATED = BRANCH_ROUTER_CONFIG.replace("area 1 stub", "area 1 totally-stub")
pipeline.process_device("branch-rtr-07", BRANCH_UPDATED, prev_config=BRANCH_ROUTER_CONFIG)

# Same core config — should be cached
pipeline.process_device("core-rtr-01", CORE_ROUTER_CONFIG)

pipeline.report()


## Summary — What You Built

| Demo | Pattern | Fixes which failure mode? |
|------|---------|--------------------------|
| 1 | **Two-stage pipeline** (Haiku extract → Sonnet write) | Audience mismatch |
| 2 | **Config fingerprinting** (hash-based change detect) | Drift — docs auto-update |
| 3 | **Multi-audience synthesis** (one extract, 3 docs) | Audience mismatch, Absence |
| 4 | **Auto change log** (before/after config diff → AI) | Absence (midnight changes) |
| 5 | **Network wiki** (plain-English queries) | Fragmentation, Tribal knowledge |

---

## The Core Architecture

```
Device configs
      ↓
Hash check (is this device changed?)
      ↓ (only if changed)
Stage 1: Haiku extracts facts → JSON       ← cheap, fast
      ↓
Stage 2: Sonnet synthesizes → Markdown     ← readable, audience-specific
      ↓
NetworkWiki (indexed docs)
      ↓
Plain-English queries → grounded answers
```

---

## Key Pricing Reality

| Task | Model | Cost per device |
|---|---|---|
| Fact extraction | Haiku | ~$0.001 |
| Doc synthesis | Sonnet | ~$0.02 |
| Change log | Sonnet | ~$0.015 |
| **Total per device** | | **~$0.036** |

With 90% cache hit rate (most devices unchanged): effective cost **~$0.004 per device per run**.
500 devices × daily runs × $0.004 = **$0.60/day** for fully automated documentation.

---

## What's Next — Chapter 14

Chapter 14 replaces the keyword search in `NetworkWiki._find_relevant()`
with **real vector embeddings** — semantic search that understands meaning, not just keywords.

Ask: *"What routing designs protect against single-provider failure?"*
and get the exact relevant documentation — even if it doesn't contain those exact words.

---
*Chapter 13 complete — Volume 2: Practical Applications*
